# Pretraining and Fine-Tuning a Language Model

> One loss and one update show how a model learns from an error. Full training extends that process across continuous corpora and many samples. The sentence `I love you` becomes input `[BOS, I, love, you]` and labels `[I, love, you, EOS]`; the model predicts four next tokens at once, and Cross-Entropy combines the four errors.
>
> One update follows this chain: `text -> token IDs -> input/labels -> logits -> loss -> gradients -> parameter update`. Pretraining and chat fine-tuning share the same chain; their main differences are data format and which positions receive supervision.
>
> **Training samples** use shifted labels for next-token answers and masks to exclude padding. **Batching and stability** use batches, Gradient Accumulation, Gradient Clipping, and Warmup to control data volume and update size. **Chat supervision** uses a Chat Template and loss mask so only assistant responses contribute. **Tools and extensions** such as Multi-Token Prediction, Transformers, and ModelScope SWIFT change targets or package the same training process behind standard interfaces.

The sequence `[BOS, I, love, you, EOS]` shifts by one position to form input `[BOS, I, love, you]` and labels `[I, love, you, EOS]`, yielding four next-token exercises. One forward pass produces logits at all four positions; Cross-Entropy compares each prediction with its answer and reduces the results to one scalar. Backpropagation starts from that scalar and produces gradients for every participating parameter.

Pretraining and chat fine-tuning use the same computation. What changes is how samples are organized and which positions retain supervision. We will begin with the smallest sample and connect input, labels, masks, and loss one by one.


## 1. A Minimal Training Sample

Suppose we have a tiny model with a vocabulary of only 5 words, and we want to train it to predict the next token.

```
Vocabulary: [BOS=0, I=1, love=2, you=3, EOS=4]

Training data is a single sentence: "I love you"
token IDs: [BOS, I, love, you, EOS]
          = [0, 1, 2, 3, 4]
```

**Training goal**: given preceding tokens, predict the next one.

```
Given [BOS]           -> predict I
Given [BOS, I]        -> predict love
Given [BOS, I, love]  -> predict you
Given [BOS, I, love, you] -> predict EOS
```

These look like 4 independent prediction tasks, but Transformer has a magic property: **it can make predictions at all positions in parallel!**

## 2. Training Data and Labels

This is the most critical concept to understand. Look at this diagram:

```
Training sentence: [BOS,  I,   love, you,  EOS]
                    [0,   1,    2,    3,    4]

          +-------------------------------------+
Input:    |  BOS  |  I    | love  |  you  |
          |  [0]  | [1]   | [2]   | [3]   |
          +-------------------------------------+
                    |  model forward
          +-------------------------------------+
Model:    | logits | logits| logits| logits|
output:   |  [0]   |  [1]  |  [2]  |  [3]  |
          +-------------------------------------+
                    |  each position predicts next token
          +-------------------------------------+
Expected: |  I     | love  |  you  |  EOS  |
labels:   |  [1]   | [2]   |  [3]  |  [4]  |
          +-------------------------------------+

Input  = sentence with last token removed
Labels = sentence with first token removed (shifted right by one)
```

**This is called teacher forcing**: instead of using the model's own predictions to continue, we feed it the **correct answers**.

Why do this? Because it allows all positions to be trained **in parallel**, without waiting for previous positions' results.

In [ ]:
# Hand-calculate one example to make the idea concrete
import torch

sentence = torch.tensor([0, 1, 2, 3, 4])  # [BOS, I, love, you, EOS]

print("Complete sentence:", sentence.tolist())
print()

# Input: remove the final token
input_ids = sentence[:-1]  # [0, 1, 2, 3]
print("Input, final token removed:", input_ids.tolist())
print("Meaning:                    [BOS, I, love, you]")
print()

# Labels: remove the first token, shifting the sequence one position to the right
target_ids = sentence[1:]   # [1, 2, 3, 4]
print("Labels, first token removed:", target_ids.tolist())
print("Meaning:                    [I, love, you, EOS]")
print()

print("Position-by-position alignment:")
for i in range(len(input_ids)):
    print(f"  Position {i}: after seeing [{', '.join(str(x) for x in input_ids[:i+1].tolist())}], predict {target_ids[i].item()}")


## 3. Cross-Entropy Loss

The model outputs a set of logits at each position (one score per word in the vocabulary). We need to compare them against the labels.

We use **Cross-Entropy Loss**:

```
For each position:
1. Convert logits to probabilities: softmax(logits)
2. Look up the probability assigned to the correct label
3. Compute -log(that probability)
4. Average across all positions
```

**Intuition**: If the correct label has probability 1.0 -> loss = -log(1.0) = 0 (perfect)
             If the correct label has probability 0.01 -> loss = -log(0.01) = 4.6 (terrible)

Let's walk through this step by step in code.

In [ ]:
# Simulate model output
import torch

vocab_size = 5
seq_len = 4  # Input length

# Simulated logits with batch size one
# A real model calculates these values; here we choose them manually for inspection
torch.manual_seed(123)
logits = torch.randn(1, seq_len, vocab_size)  # [batch=1, seq_len=4, vocab=5]
targets = torch.tensor([[1, 2, 3, 4]])          # [batch=1, seq_len=4]

print(f"Model logits shape: {logits.shape}")
print(f"targets shape:   {targets.shape}")
print()

# Inspect logits and label at position zero
print(f"Logits at position 0: {logits[0, 0].tolist()}")
print(f"Label at position 0:  {targets[0, 0].item()}")
print(f"The model must select token {targets[0, 0].item()} from five vocabulary items")


In [ ]:
# Hand-calculate loss to see where every number comes from
import torch.nn.functional as F
import math

print("=== Hand-calculate Cross-Entropy Loss ===")
print()

total_loss = 0.0
for pos in range(seq_len):
    # Logits at this position: one score per vocabulary item
    pos_logits = logits[0, pos]  # [vocab_size]
    # Correct token at this position
    correct_id = targets[0, pos].item()

    # Step 1: convert scores to probabilities with softmax
    probs = F.softmax(pos_logits, dim=-1)

    # Step 2: take the probability assigned to the correct token
    correct_prob = probs[correct_id].item()

    # Step 3: loss = -log(probability)
    pos_loss = -math.log(correct_prob)
    total_loss += pos_loss

    print(f"Position {pos}: correct token={correct_id}, probability={correct_prob:.4f}, loss={pos_loss:.4f}")

# Step 1: mean
manual_loss = total_loss / seq_len
print(f"\nMean loss over all positions: {manual_loss:.4f}")

# Compare with PyTorch cross_entropy
pt_loss = F.cross_entropy(
    logits.reshape(-1, vocab_size),  # [batch*seq_len, vocab]
    targets.reshape(-1)               # [batch*seq_len]
).item()
print(f"PyTorch cross_entropy: {pt_loss:.4f}")
print(f"Do they match? {'yes' if abs(manual_loss - pt_loss) < 1e-4 else 'no'}")


## 4. Token-Level Training

**Answer: Token-level training.**

But note: **all tokens are trained in parallel simultaneously**, not one token at a time.

```
+----------------------------------------------+
|        One forward + backward pass            |
|                                              |
|  Loss = loss(pos 0) + loss(pos 1) + ...      |
|                                              |
|  Position 0 predicts token 1                  |
|  Position 1 predicts token 2  } all parallel  |
|  Position 2 predicts token 3                  |
|  Position 3 predicts token 4                  |
|                                              |
|  Gradient = dLoss/dW is the sum of           |
|  gradients from all positions                 |
|  Parameter update <- includes learning        |
|  signals from all positions                   |
+----------------------------------------------+
```

**Why not sentence-level?**
- Sentence-level means only predicting at one position (the end) -> signal is too sparse
- For a 100-token sentence, sentence-level gives only 1 supervision signal
- Token-level gives 100 supervision signals, 100x more efficient

**But it's also not "sequential token training."** All positions are computed in parallel during a single forward pass.
This is the key reason Transformers are faster than RNNs.

## 5. Batch Training

In real training, we don't process one sentence at a time. We process a batch (e.g., 32 sentences) packed into a single matrix.

```
Batch input:
[[BOS,  I,   love, you,  EOS,  PAD,  PAD],   <- Sentence 1 (5 valid tokens)
 [BOS,  hello, world, EOS, PAD,  PAD,  PAD]]   <- Sentence 2 (4 valid tokens)

Shape: [batch_size=2, seq_len=7]
```

Loss computation: average across all sentences and all positions (excluding PAD).

```python
loss = cross_entropy(logits.reshape(-1, vocab), targets.reshape(-1), ignore_index=PAD_ID)
#                                                    ^ ignore padding positions
```

In [ ]:
# Demonstrate loss calculation for a batch
import torch
import torch.nn.functional as F

PAD_ID = 0  # Use token 0 as PAD

batch_input = torch.tensor([
    [0, 1, 2, 3, 4, 0, 0],  # [BOS, I, love, you, EOS, PAD, PAD]
    [0, 2, 4, 0, 0, 0, 0],  # [BOS, love, EOS, PAD, PAD, PAD, PAD]
])

# Labels equal the inputs shifted one position to the right
batch_target = torch.tensor([
    [1, 2, 3, 4, 0, 0, 0],  # [I, love, you, EOS, PAD, PAD, PAD]
    [2, 4, 0, 0, 0, 0, 0],  # [love, EOS, PAD, PAD, PAD, PAD, PAD]
])

print('"Batch Input:"')
print(batch_input)
print()
print("Batch labels:")
print(batch_target)
print()

# Simulate model output
batch_logits = torch.randn(2, 7, 5)  # [batch=2, seq=7, vocab=5]

# Crucial detail: ignore_index=PAD_ID excludes padded positions from loss
loss_with_ignore = F.cross_entropy(
    batch_logits.reshape(-1, 5),       # [14, 5]
    batch_target.reshape(-1),           # [14]
    ignore_index=PAD_ID
)

loss_without_ignore = F.cross_entropy(
    batch_logits.reshape(-1, 5),
    batch_target.reshape(-1)
)

print(f"Loss ignoring PAD:     {loss_with_ignore.item():.4f}")
print(f"Loss including PAD:    {loss_without_ignore.item():.4f}")
print("\nThe difference is large because PAD predictions are meaningless and should not contribute to loss.")


## 6. The Complete Training Loop

Now we combine the loss function, gradient computation, and optimizer into a complete training loop. In a real scenario, the training loop goes through the full cycle: data loading, tokenization, forward pass, loss computation, backpropagation, and parameter update.

But we haven't trained a real tokenizer yet, so we'll use a simplified setup:

- A tiny "pseudo-vocabulary" (a few dozen token IDs) to simulate tokenized input
- Manually construct the input-label offset (label = input shifted right by one, standard practice for autoregressive language models)
- Walk through the complete forward -> loss -> backward -> update flow

Although this uses simplified data, the structure is identical to real training. Understanding this means switching to a real tokenizer and data is just a matter of changing the input.

In [ ]:
# Reuse MiniGPT's core structure while retaining only the simplifications needed for teaching
import torch
import torch.nn as nn
import math

def get_sinusoidal_encoding(seq_len, d_model):
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
    )
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=4, max_seq_len=128):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len

        self.token_emb = nn.Embedding(vocab_size, d_model)
        pe = get_sinusoidal_encoding(max_seq_len, d_model)
        self.register_buffer('pe', pe)

        # Simplification: write a few blocks directly instead of using ModuleList
        self.attn1 = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn1 = nn.Sequential(
            nn.Linear(d_model, 4*d_model), nn.ReLU(), nn.Linear(4*d_model, d_model)
        )
        self.norm1a = nn.LayerNorm(d_model)
        self.norm1f = nn.LayerNorm(d_model)

        self.attn2 = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn2 = nn.Sequential(
            nn.Linear(d_model, 4*d_model), nn.ReLU(), nn.Linear(4*d_model, d_model)
        )
        self.norm2a = nn.LayerNorm(d_model)
        self.norm2f = nn.LayerNorm(d_model)

        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.shape
        x = self.token_emb(x) + self.pe[:seq_len, :]

        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device) * float('-inf'), diagonal=1)

        # Block 1
        attn_out, _ = self.attn1(x, x, x, attn_mask=mask)
        x = self.norm1a(x + attn_out)
        x = self.norm1f(x + self.ffn1(x))

        # Block 2
        attn_out, _ = self.attn2(x, x, x, attn_mask=mask)
        x = self.norm2a(x + attn_out)
        x = self.norm2f(x + self.ffn2(x))

        x = self.ln_final(x)
        return self.lm_head(x)

print('"MiniGPT ModelDefinition complete！"')


In [ ]:
# === Complete training-loop demonstration ===

# 1. Prepare synthetic data that represents tokenized text
import torch

VOCAB_SIZE = 20
PAD_ID = 0
SEQ_LEN = 16
BATCH_SIZE = 8

# Random token sequences serve as the synthetic data
train_data = torch.randint(1, VOCAB_SIZE, (100, SEQ_LEN))  # One hundred sequences
print(f"Training data: {train_data.shape}, 100 sequences of {SEQ_LEN} tokens")

# Create the model
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# 3. Training loop
import torch.nn.functional as F

NUM_EPOCHS = 5
losses = []

model.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    num_batches = 0

    for i in range(0, len(train_data), BATCH_SIZE):
        batch = train_data[i:i+BATCH_SIZE]  # [batch_size, seq_len]

        # Output projection
        input_ids = batch[:, :-1]      # Remove the final token
        target_ids = batch[:, 1:]       # Remove the first token

        # Forward
        logits = model(input_ids)  # [batch, seq_len-1, vocab_size]

        # Flatten every batch item and position before calculating loss
        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),  # [batch*(seq_len-1), vocab_size]
            target_ids.reshape(-1)            # [batch*(seq_len-1)]
        )

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {avg_loss:.4f}")

print(f"  2. Loss dropped from {losses[0]:.2f} to {losses[-1]:.2f} -- this is a real training signal on actual text.")


In [ ]:
# Visualize decreasing loss
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(losses, 'o-', markersize=8)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss curve')
plt.grid(True, alpha=0.3)
plt.show()

print("Falling loss means the model is learning next-token prediction")


## 7. Reviewing the Training Result

```
+------------------------------------------------------------------+
|        What does "token-level" really mean in LLM training?      |
+------------------------------------------------------------------+
|                                                                    |
|  Input sequence: [BOS, I, love, you, China, EOS]                  |
|                    | remove last token                             |
|  Model input:   [BOS, I, love, you, China]                        |
|                    | model forward                                 |
|  Model output:  [logits0, logits1, ..., logits4]                   |
|                       each is a [vocab_size] score vector          |
|                    | compare against labels                        |
|  Expected:      [I,     love,   you,  China, EOS]                  |
|                    |      |       |      |      |                  |
|                    +------+-------+------+------+)                 |
|                    each position computes its own loss             |
|                    |                                                |
|  Loss = mean( loss0, loss1, loss2, ..., loss4 )                    |
|                    | backpropagation                                |
|  Gradients come from all 5 positions simultaneously                |
|  -> update model parameters                                        |
|                                                                    |
|  [x] Token-level: every token position contributes to loss         |
|  [x] Parallel: all positions predicted in one forward pass         |
|  [ ] NOT sentence-level: not just the last position               |
|  [ ] NOT sequential tokens: no waiting for previous tokens        |
|                                                                    |
+------------------------------------------------------------------+
```

## 8. Training and Inference

| | During Training | During Inference/Generation |
|------|--------|------------|
| Input | Complete sentence (minus last token) | Only a prompt |
| Computation | All positions in parallel | Token by token, sequentially |
| Labels used | Ground truth (teacher forcing) | Previous self-generated token |
| Mask | Masks future tokens | Also masks future tokens |
| Loss | Computed at all token positions | No loss computation |

**Key distinction**:
- Training uses teacher forcing -> all positions in parallel -> fast
- Inference has no ground truth -> must generate one token at a time -> slow (this is the fundamental reason LLM inference is slow)

-> Next Part: Inference / autoregressive generation!


## 9. Gradients

So far we've been talking about loss, but loss is just a number. What actually drives model learning is the **gradient**.

```
loss (a single number)
    | backpropagation (backward)
gradient (one number per parameter)
    | optimizer (optimizer.step)
parameter update (model becomes slightly better)
```

This section dives into the gradient -- an often-skipped intermediate step -- to see what actually happens.

### 9.1 Backpropagation

**Intuition**: loss is the "collective result" of all model parameters. Backpropagation asks:
> "If I increase this parameter by a tiny amount, how much does loss change?"

This "rate of change" is the gradient.

Let's understand this with the simplest possible example. Suppose we have a single neuron:

```
Input x --> [weight w] --> output y = w*x
                              |
                          loss = (y - target)^2
```

**Chain Rule**:
```
dloss/dw = dloss/dy * dy/dw
         = 2(y - target) * x
```

In an LLM, this chain passes through dozens of Transformer blocks, but the principle is exactly the same --
start from loss, trace back along the computation graph step by step, multiplying the local derivative at each operation.

In [ ]:
# Manually demonstrate backpropagation through a simple network
import torch

print("=== Summary table ===")
print()

# Build the smallest possible model
w = torch.tensor([0.5], requires_grad=True)
b = torch.tensor([0.1], requires_grad=True)

x = torch.tensor([2.0])   # Input
target = torch.tensor([3.0])  # Target

print(f"Parameters: w={w.item():.2f}, b={b.item():.2f}")
print(f"Input x={x.item():.2f}, target={target.item():.2f}")
print()

# Forward
y = w * x + b              # y = 0.5*2 + 0.1 = 1.1
loss = (y - target) ** 2   # loss = (1.1 - 3)^2 = 3.61

print(f"Forward: y = w*x + b = {w.item()}*{x.item()} + {b.item()} = {y.item():.2f}")
print(f"Loss = (y - target)² = ({y.item():.2f} - {target.item():.2f})² = {loss.item():.2f}")
print()

# Backward
loss.backward()

print(f"d(loss)/dw = {w.grad.item():.4f}; increasing w by 1 changes loss locally by {w.grad.item():.4f}")
print(f"d(loss)/db = {b.grad.item():.4f}; increasing b by 1 changes loss locally by {b.grad.item():.4f}")
print()

# Verify the chain rule by hand
print("=== Tool execution ===")
print(f"∂loss/∂y = 2*(y - target) = 2*({y.item():.2f} - {target.item():.2f}) = {2*(y.item()-target.item()):.2f}")
print(f"∂y/∂w = x = {x.item():.2f}")
print(f"∂y/∂b = 1")
print(f"∂loss/∂w = ∂loss/∂y * ∂y/∂w = {2*(y.item()-target.item()):.2f} * {x.item():.2f} = {2*(y.item()-target.item())*x.item():.2f}")
print(f"PyTorch d(loss)/dw = {w.grad.item():.4f}, which matches the manual result")


In [ ]:
# Inspect gradient flow through MiniGPT
import torch
import torch.nn.functional as F

VOCAB_SIZE = 20
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)

dummy_input = torch.randint(1, VOCAB_SIZE, (2, 16))   # [batch=2, seq=16]
dummy_target = torch.randint(1, VOCAB_SIZE, (2, 15))  # [batch=2, seq=15]

# Forward
logits = model(dummy_input[:, :-1])
loss = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    dummy_target.reshape(-1)
)

# Backward
model.zero_grad()
loss.backward()

print("=== MiniGPT Main Components ===")
print()
print(f"{'Layer':<40s} {'Gradient norm':>12s} {'Parameter shape':>18s}")
print("-" * 72)
total_grad_norm = 0
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        total_grad_norm += grad_norm ** 2
        param_shape = str(list(param.shape))
        print(f"{name:<40s} {grad_norm:>12.6f}  {param_shape:>18s}")

total_grad_norm = total_grad_norm ** 0.5
print("-" * 72)
print(f"{'Total gradient norm (L2)':<40s} {total_grad_norm:>12.6f}")
print()
print('"observe:"')
print("  1. Every parameter has a gradient, so backpropagation carried the loss signal through every layer")
print("  2. lm_head often has a large gradient because it is closest to the loss")
print("  3. Embedding gradients are smaller because the signal has passed through many layers")
print("  4. Residual connections keep gradients from disappearing, making Transformers easier to train")


### 9.2 Token-Level Gradients

Earlier we said "every token contributes to loss." But wait: **do all tokens contribute equally?**

**No!** Consider this example:

```
Sentence: "The capital of France is Paris"
            |--easy--|  |med|  |key info|
            (low loss)         (high loss)
```

- Common words like "of", "is" are learned quickly -> low loss -> **small gradients**
- Key content words like "Paris" -> high loss -> **large gradients**

**This means the learning signal is primarily driven by "hard tokens," while easy tokens barely contribute gradients.**

This leads to the core problem found in RL training: **Advantage Collapsing** -- most rollouts have advantages near 0, resulting in weak gradient signals.

In [ ]:
# Show that different token positions have different losses and therefore different gradients
import torch
import torch.nn.functional as F

print("=== Token-level loss analysis ===")
print()

VOCAB_SIZE = 20
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)

# Construct a sequence whose first half follows an easy pattern and whose second half is random
# Easy pattern repeats [1, 2, 3, 1, 2, 3, 1, 2, 3]
easy_part = torch.tensor([1, 2, 3, 1, 2, 3, 1, 2, 3])
# Random portion
hard_part = torch.randint(10, VOCAB_SIZE, (7,))
sentence = torch.cat([easy_part, hard_part])  # seq_len=16
batch = sentence.unsqueeze(0)  # [1, 16]

input_ids = batch[:, :-1]      # [1, 15]
target_ids = batch[:, 1:]       # [1, 15]

print(f"First half, easy pattern: {easy_part.tolist()}")
print(f"Second half, random tokens: {hard_part.tolist()}")
print()

# Forward pass; retain the graph so each position can be examined
logits = model(input_ids)  # [1, 15, vocab_size]

# Calculate loss at every position
print("Loss at each token position:")
print(f"{'Position':>4s} {'Token':>6s} {'Loss':>10s} {'Region'}")
print("-" * 40)

for pos in range(15):
    pos_logits = logits[0, pos]  # [vocab_size]
    pos_target = target_ids[0, pos]
    pos_loss = F.cross_entropy(pos_logits.unsqueeze(0), pos_target.unsqueeze(0)).item()
    region = "easy" if pos < 8 else "hard"
    print(f"{pos:>4d} {pos_target.item():>6d} {pos_loss:>10.4f}  {region}")

# Compare mean loss in the easy and hard regions
logits_flat = logits.reshape(-1, VOCAB_SIZE)
targets_flat = target_ids.reshape(-1)
all_losses = F.cross_entropy(logits_flat, targets_flat, reduction='none')

easy_avg = all_losses[:9].mean().item()
hard_avg = all_losses[9:].mean().item()

print()
print(f"Mean easy-region loss: {easy_avg:.4f}")
print(f"Mean hard-region loss: {hard_avg:.4f}")
print(f"Hard-region loss is {hard_avg/easy_avg:.2f}x the easy-region loss")
print()
print("Hard tokens create larger gradients and drive more learning.")
print("This is why RL training tracks which rollouts and tokens actually contribute gradients.")


In [ ]:
# Advanced view: visualize token-level gradient distribution in an LLM
import torch

print("=== Input token IDs ===")
print()

# Simulate the gradient norm of every token in one sentence
torch.manual_seed(42)

# Simulate losses for twenty tokens, small at the beginning and larger later
token_losses = torch.tensor([0.1, 0.15, 0.2, 0.15, 0.1,
                              0.3, 0.5, 0.8, 1.2, 1.5,
                              2.0, 2.5, 2.8, 3.0, 3.2,
                              3.5, 3.8, 4.0, 4.2, 4.5])

tokens = ["BOS", "I", "am", "an", "AI",
          "today", "the", "weather", "is", "pleasant",
          "quantum", "entanglement", "is", "nonlocal", "in",
          "physical", "systems", "today", ",", "EOS"]

# Simplification: assume gradient contribution is proportional to loss
grad_contrib = token_losses / token_losses.sum() * 100

print("Token-level gradient contributions:")
print(f"{'Token':<8s} {'Loss':>8s} {'Gradient share':>12s} {'Visualization'}")
print("-" * 60)

threshold = 5.0  # Treat a token above 5% as high contribution
high_count = 0
for i in range(len(tokens)):
    bar_len = int(grad_contrib[i].item() * 3)
    bar = "█" * bar_len
    marker = " ★" if grad_contrib[i] > threshold else ""
    if grad_contrib[i] > threshold:
        high_count += 1
    print(f"{tokens[i]:<8s} {token_losses[i].item():>8.2f} {grad_contrib[i].item():>10.1f}% {bar}{marker}")

print()
print(f"Tokens with gradient contribution above {threshold}%: {high_count}/{len(tokens)}")
print(f"Those {high_count} tokens contribute {grad_contrib[-high_count:].sum().item():.1f}% of the gradient")
print()
print('"Key insight:"')
print("  1. Early high-frequency tokens contribute almost no gradient")
print("  2. Later content and difficult tokens contribute most of it")
print("  3. Training efficiency is therefore determined by a small set of difficult tokens")
print("  4. Shuffle-R1 uses PTS and ABS to address exactly this issue in RL")


### 9.3 Gradient Clipping

**Problem**: Sometimes a batch produces extremely large gradients (e.g., encountering a never-before-seen pattern). If the optimizer takes a full step with this gradient, parameters fly off -- loss becomes NaN, and training crashes.

**Solution**: Gradient clipping. Set an upper bound:

```
If gradient L2 norm > max_norm:
    Scale all gradients proportionally so norm = max_norm
Otherwise:
    Keep unchanged
```

```
       Gradient explosion
       ^
       |    /\             after clipping
       |   /  \           - - - -  max_norm
       |  /    \--       /
       | /             /
       |/-------     /
       +---------------> training step
       Without clipping, that spike would crash the model
```

Gradient clipping is almost always used in LLM training, typically with `max_norm=1.0`.

In [ ]:
# Demonstrate gradient clipping
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=== Summary table ===")
print()

# Build a small network and deliberately create an exploding gradient
linear = nn.Linear(10, 1)

x = torch.randn(5, 10)
target = torch.randn(5, 1) * 100  # Scale the target to create a large gradient

# Without clipping
loss = F.mse_loss(linear(x), target)
loss.backward()

raw_grad_norm = sum(p.grad.norm().item() ** 2 for p in linear.parameters()) ** 0.5
print(f"Gradient norm without clipping: {raw_grad_norm:.4f}")

# Reset gradients
linear.zero_grad()

# Apply clipping
loss = F.mse_loss(linear(x), target)
loss.backward()
max_norm = 1.0
nn.utils.clip_grad_norm_(linear.parameters(), max_norm)
clipped_grad_norm = sum(p.grad.norm().item() ** 2 for p in linear.parameters()) ** 0.5

print(f"Gradient norm after clipping: {clipped_grad_norm:.4f}, limit={max_norm}")
print()
print(f"The oversized gradient is clipped to {max_norm}, preventing an unstable update")
print()

# Add clipping to a real training loop
print("Practical training-loop snippet:")
print("```python")
print("loss.backward()")
print("torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)")
print("optimizer.step()")
print("```")
print()
print("This clipping line is a safety belt for LLM training:")
print("usually unobtrusive, but protective when a gradient spike occurs.")


### 9.4 Gradient Accumulation

**Problem**: LLM training needs large batches (e.g., 512), but your GPU can only fit batch=4. What do you do?

**Key observation**:
```
batch=8 gradient = batch=4 gradient + batch=4 gradient
                      ^ first batch       ^ second batch
```

Gradients can be "accumulated." So:

```
Target batch = 512, GPU can only run 32 at a time
-> Run 512/32 = 16 small batches, accumulating gradients without updating
-> After 16 runs, the gradient is equivalent to batch=512
-> Then call optimizer.step()
```

**Tradeoff**: Training becomes slower (16 forward passes per update), but at least it's possible.

**Difference from regular small batches**:
- Regular small batch: each forward -> backward -> step, the batch is genuinely small
- Gradient accumulation: multiple forward -> backward -> final step, **equivalent to a large batch**

In [ ]:
# Demonstrate gradient accumulation
import torch
import torch.nn.functional as F

print("=== Summary table ===")
print()

VOCAB_SIZE = 20
model_small = MiniGPT(VOCAB_SIZE, d_model=32, num_heads=2, num_layers=1)
model_large = MiniGPT(VOCAB_SIZE, d_model=32, num_heads=2, num_layers=1)

# Copy identical parameters
model_large.load_state_dict(model_small.state_dict())

# Synthetic data
all_data = torch.randint(1, VOCAB_SIZE, (16, 16))  # Sixteen samples in total

# Method A: train directly with one large batch of 16
opt_large = torch.optim.SGD(model_large.parameters(), lr=0.01)

input_large = all_data[:, :-1]
target_large = all_data[:, 1:]
logits_large = model_large(input_large)
loss_large = F.cross_entropy(
    logits_large.reshape(-1, VOCAB_SIZE),
    target_large.reshape(-1)
)
opt_large.zero_grad()
loss_large.backward()

# Save the large-batch gradients
grads_large = {name: p.grad.clone() for name, p in model_large.named_parameters() if p.grad is not None}
opt_large.step()

# Method B: use batches of four and accumulate four steps
opt_small = torch.optim.SGD(model_small.parameters(), lr=0.01)
opt_small.zero_grad()

ACCUM_STEPS = 4
small_batch_size = 4
for step in range(ACCUM_STEPS):
    start = step * small_batch_size
    end = start + small_batch_size
    mini_batch = all_data[start:end]

    input_small = mini_batch[:, :-1]
    target_small = mini_batch[:, 1:]
    logits_small = model_small(input_small)
    loss_small = F.cross_entropy(
        logits_small.reshape(-1, VOCAB_SIZE),
        target_small.reshape(-1)
    )

    # Divide loss by the accumulation count to preserve the gradient scale
    (loss_small / ACCUM_STEPS).backward()
    print(f"  Accumulation step {step+1}/{ACCUM_STEPS}: loss={loss_small.item():.4f}; gradients added, no update yet")

print()

# Save gradients accumulated from small batches
grads_small = {name: p.grad.clone() for name, p in model_small.named_parameters() if p.grad is not None}
opt_small.step()

# Compare whether the two methods produce matching gradients
print("=== Three Signal Granularity Levels Compared ===")
all_close = True
for name in grads_large:
    diff = (grads_large[name] - grads_small[name]).norm().item()
    status = "[ok]" if diff < 1e-4 else "[x]"
    if diff >= 1e-4:
        all_close = False
    print(f"  {name:<30s} difference={diff:.8f} {status}")

print()
if all_close:
    print('Conclusion')
    print("Four forward passes with batch size four reproduce a batch size of sixteen")
else:
    print("Sequential mini-batches can differ slightly in networks that use BatchNorm")
    print("For LLMs using LayerNorm, the two calculations are equivalent")


### 9.5 Gradient Flow in an LLM

Let's connect everything we've learned and trace the complete lifecycle of a gradient during LLM training:

```
+------------------------------------------------------------------+
|              Complete LLM Gradient Flow Diagram                    |
+------------------------------------------------------------------+
|                                                                    |
|  [Data]                                                            |
|    |                                                                |
|  [Forward: Input -> Embedding -> Transformer Blocks -> LM Head]    |
|    |                                                                |
|  [Loss: Cross-Entropy, one loss per token]                         |
|    |                                                                |
|    |  <- Key observation 1: Token-level gradients                  |
|    |     Easy tokens (of/is) -> low loss -> small gradients        |
|    |     Hard tokens (Paris) -> high loss -> large gradients       |
|    |     Training is mainly driven by "hard tokens"                |
|    |                                                                |
|  [Backward: gradients flow back from LM Head]                      |
|    |                                                                |
|    +-- LayerNorm: normalization, stable gradients                  |
|    +-- FFN: two Linear layers, gradients may be large              |
|    +-- Attention: QKV projections + Output projection              |
|    |     |                                                          |
|    |     +-- <- Key observation 2: attention head gradient diff.   |
|    |           Some heads have large gradients (active),            |
|    |           some have small gradients (redundant)               |
|    +-- Residual connection -> gradient shortcut, no vanishing      |
|    |                                                                |
|    | Back to Embedding layer (smallest gradients)                  |
|    |                                                                |
|  [Gradient Processing]                                             |
|    +-- Gradient Clipping: prevent explosion, max_norm~1.0          |
|    +-- Gradient Accumulation: small GPU simulates large batch      |
|    +-- (RL-specific) PTS+ABS: filter high-contribution rollouts    |
|    |                                                                |
|  [Parameter Update: optimizer.step()]                              |
|    |                                                                |
|  [Next batch, repeat]                                              |
|                                                                    |
+------------------------------------------------------------------+
```

**Three most important takeaways**:

1. **Residual connections = gradient highways**: Without them, gradients in deep Transformers would vanish (like in RNNs).
   Residual connections let gradients "skip" FFN and Attention, flowing directly backward.

2. **Token-level gradients are uneven**: Easy words barely contribute gradients; training is driven by hard words.
   This explains why LLMs improve slowly on "knowledge-intensive" tasks --
   knowledge words are a small fraction, so gradient signals are sparse.

3. **RL training amplifies gradient unevenness**: In SFT at least every token has a clear label,
   but in RL many rollouts have advantages near 0 -> even sparser gradient signals.
   Shuffle-R1's PTS+ABS was designed to address this.

In [ ]:
# Visualize how residual connections protect gradient flow
print("=== Summary table ===")
print()

print("Without residual connections, as in a traditional deep network:")
print("  Input → Layer1 → Layer2 → ... → Layer32 → Output")
print("  Gradient path: Output -> Layer32 -> ... -> Layer1 -> Input")
print("  After multiplication through 32 layers, gradients can decay to zero")
print()

print("With residual connections, as in a Transformer:")
print("  Input → [Layer1 + Input] → [Layer2 + prev] → ... → Output")
print("  Gradients have two routes:")
print("    Main route: Output -> Layer32 -> ... -> Layer1 -> Input, which may decay")
print("    Shortcut: Output -> Input, skipping all layers directly")
print()
print("The shortcut guarantees that at least part of the gradient reaches lower layers without attenuation.")
print("This is one reason Transformers can be trained with more than one hundred layers.")
print()

# Simulate gradient attenuation at different depths
depths = [1, 4, 8, 16, 32, 64]

print("Simulation: begin with gradient 1.0 and apply attenuation through different depths")
print(f"{'':>8} {'unmasked':>30}  {'masked':>30}")
print("-" * 32)

decay_per_layer = 0.95  # Each layer attenuates the main path by 5%
skip_ratio = 0.3         # The residual path carries 30%

for d in depths:
    no_skip = decay_per_layer ** d
    with_skip = no_skip * (1 - skip_ratio) + skip_ratio
    print(f"{d:>6d}  {no_skip:>12.6f}  {with_skip:>12.6f}")

print()
print('Conclusion')
print("The deeper the network, the more valuable the residual connection becomes.")


### 9.6 Gradient Summary

| Concept | One-liner | Why it matters |
|:---|:---|:---|
| **Backpropagation** | Loss -> chain rule -> gradient per parameter | Foundation of training; no learning without it |
| **Token-level gradients** | Hard tokens get large gradients, easy tokens near 0 | Explains why training efficiency is driven by "hard samples" |
| **Gradient clipping** | Scale down proportionally when norm > max_norm | Safety belt against training crashes |
| **Gradient accumulation** | Accumulate gradients from multiple small batches, update once | Lifesaving technique for training large models on small GPUs |
| **Residual connections** | Gradients can skip layers and flow directly backward | Fundamental reason Transformers can be 100+ layers deep |
| **RL gradient sparsity** | Even fewer effective gradients in RL training | Motivation for work like Shuffle-R1 |

**Gradients are the "translator" between loss and parameter updates** --
they translate "where this batch performed poorly" into "how each parameter should be adjusted."
Understanding gradients is essential to truly understanding how models learn.

## 10. Tokenizing Conversation Data

> **Analogy**: You want to write a letter to a friend. You have "what you want to say" in your head, but the post office only accepts "words written on paper." Chat Template is that "format for turning thoughts into a letter" -- you write "Dear So-and-so" first, then the body, then sign off.
> LLMs work the same way: your conversational messages can't be fed directly to the model; they must first be "formatted" into tokens via Chat Template.

### 10.1 Training Data Format

Real LLM training data comes in JSONL format (one JSON object per line), which is the OpenAI API-compatible standard:

```jsonl
{"messages": [{"role": "system", "content": "You are a math assistant"}, {"role": "user", "content": "1+1=?"}, {"role": "assistant", "content": "1+1=2"}]}
{"messages": [{"role": "user", "content": "Hello"}, {"role": "assistant", "content": "Hello! How can I help?"}]}
{"messages": [{"role": "system", "content": "You are a translator"}, {"role": "user", "content": "Hello"}, {"role": "assistant", "content": "Ni hao"}]}
```

**Core question**: how does the JSON object above become the `input_ids` and `labels` that the model sees?

We need to understand three things:
1. **Concatenation**: How is the messages list concatenated into a continuous token sequence?
2. **Segmentation**: Which tokens are "context for the model to read" vs. "answers for the model to learn"?
3. **Loss**: How are special markers (like `<|im_start|>`) handled during loss computation?

Let's walk through this step by step using the `transformers` library.

In [ ]:
# ============================================================
# Use the real transformers library to inspect what a Chat Template does
# ============================================================
print('Real tokenizer: GPT-2 byte-level BPE')

# Try to load the Qwen2.5 tokenizer; the first run downloads it automatically
try:
    from transformers import AutoTokenizer
    
    # The Qwen2.5-0.5B tokenizer is small, about 30 MB
    # Its ChatML-style template is also used across the Qwen and DeepSeek families
    MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
    
    print(f"Loading tokenizer: {MODEL_NAME}")
    print("The first run downloads about 30 MB; please wait.\n")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    
    # First inspect the tokenizer's special tokens
    print("=== Tokenizer special tokens ===")
    print(f"  bos_token:        {repr(tokenizer.bos_token)} → id={tokenizer.bos_token_id}")
    print(f"  eos_token:        {repr(tokenizer.eos_token)} → id={tokenizer.eos_token_id}")
    print(f"  pad_token:        {repr(tokenizer.pad_token)} → id={tokenizer.pad_token_id}")
    print(f"Vocabulary size: {len(tokenizer)}")
    print()
    
    # The chat template itself is simply a Jinja2 template string
    print("=== Chat Template, a Jinja2 template ===")
    ct = tokenizer.chat_template
    if ct:
        # Display only the first 500 characters
        print(ct[:500])
        print("...")
    print()
    
    # ============================================================
    # Core demonstration: what does apply_chat_template actually do?
    # ============================================================
    messages = [
        {"role": "system", "content": "You are a helpful assistant that extracts information. Answer briefly based on the text above."},
        {"role": "user", "content": "1+1=?"},
        {"role": "assistant", "content": "1+1=2"},
    ]
    
    print("=== Input: messages list ===")
    import json
    print(json.dumps(messages, ensure_ascii=False, indent=2))
    print()
    
    # Step 1: tokenize=False returns human-readable rendered text
    print("=== Step 1: apply_chat_template(tokenize=False) renders text ===")
    rendered_text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False,            # Render text without tokenizing it yet
        add_generation_prompt=False # Training data does not need a generation prompt
    )
    print("After allowing special token:")
    print(repr(rendered_text))
    print()
    print("Observations:")
    print(rendered_text)
    print()
    
    # Step 2: tokenize=True returns input_ids directly
    print("=== Step 2: apply_chat_template(tokenize=True) returns token IDs ===")
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_tensors="pt"  # Return a PyTorch tensor
    )
    print(f"input_ids shape: {input_ids.shape}")  # [batch=1, seq_len]
    print(f"  input_ids values: {input_ids[0].tolist()}")
    print(f"  Sequence length: {len(input_ids[0])}")
    print()
    
    # Step 3: look up each token's ID in the vocabulary.
    print('Inspect each token:')
    print(f"{'Position':<5s} {'Token ID':>8s} {'Decoded text':<30s} {'Note'}")
    print("-" * 75)
    
    for i, tid in enumerate(input_ids[0].tolist()):
        decoded = tokenizer.decode([tid])
        # Count new tokens learned this round
        if tid == tokenizer.bos_token_id:
            note = "BOS, beginning marker"
        elif tid == tokenizer.eos_token_id:
            note = "EOS, ending marker"
        elif tid >= len(tokenizer) - 20:  # Special tokens are commonly near the end of the vocabulary
            note = "← special token"
        elif tid == 151644:  # Qwen <|im_start|>
            note = "<|im_start|> special marker"
        elif tid == 151645:  # Qwen <|im_end|>
            note = "<|im_end|> special marker"
        else:
            note = ""
        print(f"{i:<5d} {tid:>8d} {decoded:<30s} {note}")
    
    print()
    print("  1. The text becomes one continuous token sequence, not three independent arrays")
    print("  2. <|im_start|> and <|im_end|> separate system, user, and assistant messages")
    print("  3. The sequence contains the system prompt, user question, and assistant answer")
    print("  4. The model reads the entire sequence at once under a causal attention mask")

except ImportError:
    print("transformers library not installed. Run: pip install transformers")
    print()
    print("Manually simulating Qwen2.5 chat-template behavior ...")
    print()
    
    # Simulated Qwen2.5 token IDs using the real special-token values
    print("Qwen2.5 Chat Template rendering rules in ChatML format:")
    print('  <|im_start|>system')
    print('  {system_content}')
    print('  <|im_end|>')
    print('  <|im_start|>user')
    print('  {user_content}')
    print('  <|im_end|>')
    print('  <|im_start|>assistant')
    print('  {assistant_content}')
    print('  <|im_end|>')
    print()
    
    print("Implementing a simplified version ourselves ...")
    
except Exception as e:
    print(f"Reason: {e}")
    print("Running the simplified demonstration ...")


### 10.2 Two Chat Template Implementations

Above we used `tokenizer.apply_chat_template()` to do everything in one step. But what does it actually do internally? Are there any hidden operations?

Let's do a **controlled experiment**:

| Method | How it works |
|------|--------|
| Method 1 (Official) | Call `tokenizer.apply_chat_template(messages)` |
| Method 2 (Manual) | Manually concatenate strings in ChatML format, then `tokenizer.encode()` |

**If both produce the same result, it proves that `apply_chat_template` has no magic internally -- it's just string concatenation + tokenization.**

Using the same example:
```json
{"messages": [
    {"role": "system", "content": "You are a math assistant"},
    {"role": "user", "content": "1+1=?"},
    {"role": "assistant", "content": "1+1=2"}
]}
```

ChatML format template rules (this is what the Jinja2 template does):
```
Each message -> <|im_start|>{role}\n{content}<|im_end|>\n
```

Broken into steps:

```
Step 1: system message
  <|im_start|>system\nYou are a math assistant<|im_end|>\n
  \---+---/  \-+-/  \--+-/  \------+---/  \--+--/ \+/
  special    role   newline  content   special  newline

Step 2: user message (immediately after system)
  ...<|im_end|>\n<|im_start|>user\n1+1=?<|im_end|>\n
     \--system message end--/  \--user message start---/

Step 3: assistant message (immediately after user)
  ...<|im_end|>\n<|im_start|>assistant\n1+1=2<|im_end|>\n
     \--user message end----/  \----assistant message-----/
```

**Key point**: All messages are concatenated into **a single continuous text**, with no spaces, line break separators, or array markers.
The model uses special tokens like `<|im_start|>` and `<|im_end|>` to identify message boundaries.

Below we'll use both methods side by side, printing intermediate steps, then comparing item by item:

In [ ]:
# ============================================================
# 10.3 Controlled experiment: manual concatenation versus the official API
# ============================================================
# apply_chat_template contains no hidden magic; it concatenates strings according to ChatML
# Reproduce the concatenation manually and compare every result with the official path

# If the real transformers tokenizer is unavailable, use an offline simplified tokenizer
if "tokenizer" not in globals():
    print('Real tokenizer unavailable; using SimpleChatTokenizer for an offline demonstration.')

    class SimpleChatTokenizer:
        """Minimal ChatML tokenizer showing that a template is string concatenation plus tokenization."""
        def __init__(self):
            self.special = {"<|im_start|>": 100001, "<|im_end|>": 100002}
            self.vocab = {"\n": 10}
            self.reverse = {10: "\n", 100001: "<|im_start|>", 100002: "<|im_end|>"}

        def convert_tokens_to_ids(self, token):
            return self.special.get(token, self.vocab.get(token, -1))

        def apply_chat_template(self, messages, tokenize=False):
            text = "".join(
                f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n"
                for m in messages
            )
            return self.encode(text, add_special_tokens=False) if tokenize else text

        def encode(self, text, add_special_tokens=False):
            ids = []
            i = 0
            while i < len(text):
                matched = False
                for tok, tid in self.special.items():
                    if text.startswith(tok, i):
                        ids.append(tid)
                        i += len(tok)
                        matched = True
                        break
                if matched:
                    continue
                ch = text[i]
                if ch not in self.vocab:
                    self.vocab[ch] = 1000 + len(self.vocab)
                    self.reverse[self.vocab[ch]] = ch
                ids.append(self.vocab[ch])
                i += 1
            return ids

        def decode(self, ids):
            return "".join(self.reverse[i] for i in ids)

    tokenizer = SimpleChatTokenizer()

print("=" * 70)
print("Controlled experiment: manual concatenation versus apply_chat_template")
print("=" * 70)

# ============================================================
# Prepare dialogue data
# ============================================================
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 1+1?"},
    {"role": "assistant", "content": "1+1 equals 2."},
]

print("\nOriginal dialogue:")
for i, msg in enumerate(messages):
    print(f"  [{i}] {msg['role']}: {msg['content']}")

# ============================================================
# Inspect the special-token IDs that form the ChatML skeleton
# ============================================================
IM_START_ID = tokenizer.convert_tokens_to_ids("<|im_start|>")
IM_END_ID   = tokenizer.convert_tokens_to_ids("<|im_end|>")
NEWLINE_ID  = tokenizer.convert_tokens_to_ids("\n")

print("\nSpecial-token IDs:")
print(f"  '<|im_start|>' -> ID = {IM_START_ID}")
print(f"  '<|im_end|>'   -> ID = {IM_END_ID}")
print(f"  '\\n'           -> ID = {NEWLINE_ID}")

# ============================================================
# Method 1: official apply_chat_template API
# ============================================================
print("\n" + "=" * 70)
print("Method 1: official apply_chat_template API")
print("=" * 70)

# tokenize=False returns a human-readable string
official_text = tokenizer.apply_chat_template(messages, tokenize=False)
# tokenize=True returns a token-ID list
official_ids = tokenizer.apply_chat_template(messages, tokenize=True)

print("\nOfficial rendered text, repr form:")
print(f"  {repr(official_text)}")

print("\nOfficial rendered text, readable form:")
print(f"  {official_text}")

print("\nOfficial token IDs, complete list:")
print(f"  {official_ids}")

# Decode the official result token by token
print("\nOfficial result decoded token by token:")
print(f"  {'Position':<5s} {'ID':>7s}  {'Decoded':<18s}  {'Marker'}")
print(f"  {'-'*55}")
for i, tid in enumerate(official_ids):
    decoded = repr(tokenizer.decode([tid]))
    marker = '"<-- specialtoken"' if tid in (IM_START_ID, IM_END_ID) else ""
    print(f"  [{i:>3d}] {tid:>7d}  {decoded:<18s}  {marker}")
print(f"  Total tokens: {len(official_ids)}")

# ============================================================
# Method 2: concatenate manually, reproducing apply_chat_template internals
# ============================================================
print("\n" + "=" * 70)
print("Method 2: manual concatenation in ChatML format")
print("=" * 70)

# ChatML rule:
# Each message is <|im_start|>role\ncontent<|im_end|>\n
# Concatenate all messages in order to form one continuous string
# This is exactly what the Jinja2 template inside apply_chat_template does

print("ChatML rule: format every message with role and content markers, then concatenate all segments in order.")

all_text = ""    # Accumulated text
all_ids = []     # Accumulated token IDs

print("Starting step-by-step concatenation ...\n")

for step, msg in enumerate(messages):
    role = msg["role"]
    content = msg["content"]

    # Construct this segment in ChatML format
    segment_text = f"<|im_start|>{role}\n{content}<|im_end|>\n"
    # Encode the segment without adding another set of special tokens
    segment_ids = tokenizer.encode(segment_text, add_special_tokens=False)

    before_len = len(all_ids)
    all_text += segment_text      # Concatenate strings
    all_ids.extend(segment_ids)   # Concatenate token IDs

    print(f"--- Step {step+1}: append the {role} message ---")
    print("  Segment text:")
    print(f"    {repr(segment_text)}")
    print(f"  Segment token IDs, {len(segment_ids)} total:")
    print(f"    {segment_ids}")
    print(f"  Accumulated token count: {before_len} -> {len(all_ids)}, +{len(segment_ids)}")
    print("  Current accumulated-text repr:")
    print(f"    {repr(all_text)}")
    print()

# ============================================================
# Core comparison: verify official and manual results item by item
# ============================================================
print("=" * 70)
print("Comparison: official API versus manual concatenation")
print("=" * 70)

# Check 1: are the text strings identical?
print("\n[Check 1] Compare text strings")
print(f"  Official text == manual text: {official_text == all_text}")
if official_text != all_text:
    print(f"  Official length = {len(official_text)}, manual length = {len(all_text)}")
    # Locate the first differing character
    for i, (a, b) in enumerate(zip(official_text, all_text)):
        if a != b:
            print(f"  First difference at position {i}: official={repr(a)} versus manual={repr(b)}")
            print(f"     Official context: ...{repr(official_text[max(0,i-10):i+10])}...")
            print(f"     Manual context:   ...{repr(all_text[max(0,i-10):i+10])}...")
            break
else:
    print("  The strings are identical: manual concatenation matches the official API.")

# Check 2: are the token-ID sequences identical?
print("\n[Check 2] Compare token-ID sequences")
manual_ids_list = list(all_ids)
print(f"  Official IDs == manual IDs: {official_ids == manual_ids_list}")
if official_ids != manual_ids_list:
    print(f"  Official has {len(official_ids)} tokens; manual has {len(manual_ids_list)}")
    for i in range(min(len(official_ids), len(manual_ids_list))):
        if official_ids[i] != manual_ids_list[i]:
            print(f"  First difference at position {i}:")
            print(f"     Official ID={official_ids[i]} -> {repr(tokenizer.decode([official_ids[i]]))}")
            print(f"     Manual ID={manual_ids_list[i]} -> {repr(tokenizer.decode([manual_ids_list[i]]))}")
            break
else:
    print(f"  All {len(official_ids)} token IDs are identical.")

# Check 3: do both sequences decode back to the same text?
print("\n[Check 3] Decode both sequences")
off_decoded = tokenizer.decode(official_ids)
man_decoded = tokenizer.decode(manual_ids_list)
print(f"  Official IDs decode to: {repr(off_decoded)}")
print(f"  Manual IDs decode to:   {repr(man_decoded)}")
print(f"  Identical: {off_decoded == man_decoded}")

# ============================================================
# Final summary
# ============================================================
print("\n" + "=" * 70)
print('Conclusion')
print("=" * 70)
print("A chat template iterates over messages, formats each role and content segment, concatenates them, and tokenizes the resulting continuous text.")


### 10.3 Labels and Special Tokens

Now we have `input_ids` (all tokens the model sees), but training also needs `labels` (telling the model "which tokens should you learn").

**Analogy**: An English exam gives you a reading passage + questions + reference answers.
- The passage (system prompt) -> you read it, but don't need to memorize it -> labels = IGNORE
- The questions (user message) -> you read them, but don't need to recite them -> labels = IGNORE
- The answers (assistant message) -> this is what you need to learn to write -> labels = real token IDs
- Punctuation/formatting (special tokens) -> these are formatting symbols, don't need to predict -> labels = IGNORE

```
input_ids: [151644, 8948, 198, 9942, 10603, 107659, 113738, 151645, 198, 
            151644, 872, 198, 16, 17, 18, 19, 20, 151645, 198,
            151644, 78191, 198, 16, 17, 18, 19, 18, 151645, 198]
           |-- system frame+content --||-- user frame+content --||- assistant content -|

labels:    [-100, -100, -100, -100, -100, -100, -100, -100, -100,
            -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
            -100, -100, -100, 16,   17,   18,   19,   18,  -100, -100]
           ^ All ignored in loss (system+user+special tokens)    ^ Only here   ^ Also ignored
```

**Why use -100 for labels?**
PyTorch's `CrossEntropyLoss` has an `ignore_index` parameter, defaulting to `-100`.
All positions where the label equals `ignore_index`: no loss is computed, no gradient is produced.

```
CrossEntropyLoss(ignore_index=-100) behavior:
  label = 5    -> normal computation: loss = -log(pred[5])
  label = -100 -> skip: loss = 0, grad = 0
```

Below we construct labels in code and verify that loss computation indeed skips the -100 positions:

In [ ]:
# ============================================================
# Construct labels and verify the low-level behavior of ignore_index
# ============================================================
import torch
import torch.nn.functional as F

print("=== Label construction and ignore_index verification ===\n")

# Reuse the vocabulary and tokenizer above
vocab = {
    "<|im_start|>": 151644, "<|im_end|>": 151645,
    "system": 8948, "user": 872, "assistant": 78191,
    "you": 9942, "are": 10603, "math": 107659, "assistant": 113738,
    "1": 16, "+": 17, "2": 18, "=": 19, "?": 20, "\n": 198,
}
id_to_word = {v: k for k, v in vocab.items()}

def encode(text):
    tokens = []
    i = 0
    while i < len(text):
        matched = None
        for word in sorted(vocab.keys(), key=lambda x: -len(x)):
            if text[i:].startswith(word):
                matched = word
                break
        if matched:
            tokens.append(vocab[matched])
            i += len(matched)
        else:
            tokens.append(0)
            i += 1
    return tokens

messages = [
        {"role": "system", "content": "You are a helpful assistant that extracts information. Answer briefly based on the text above."},
    {"role": "user", "content": "1+1=?"},
    {"role": "assistant", "content": "1+1=2"},
]

IM_START = "<|im_start|>"
IM_END = "<|im_end|>"
IGNORE = -100  # PyTorch default ignore_index

# ============================================================
# Step 1: construct input_ids
# ============================================================
print("Step 1: construct input_ids")
all_text = ""
for msg in messages:
    all_text += f"{IM_START}{msg['role']}\n{msg['content']}{IM_END}\n"

input_ids = torch.tensor([encode(all_text)])
print(f"  input_ids: {input_ids[0].tolist()}")
print()

# ============================================================
# Step 2: construct labels while tracking the origin of every token
# ============================================================
print("Step 2: construct labels, annotating every token")
print()

labels = torch.full_like(input_ids, IGNORE)  # Initialize every position as ignored

pos = 0
for msg in messages:
    role = msg["role"]
    content = msg["content"]

    # Header <|im_start|>role\n remains ignored
    header = f"{IM_START}{role}\n"
    hlen = len(encode(header))
    # These hlen tokens are already IGNORE
    pos += hlen

    # Concatenate
    cids = encode(content)
    if role == "assistant":
        # Only assistant-content positions receive real token labels
        for cid in cids:
            labels[0, pos] = cid
            pos += 1
    else:
        # System and user content stays ignored
        pos += len(cids)

    # Footer <|im_end|>\n remains ignored
    footer = f"{IM_END}\n"
    flen = len(encode(footer))
    pos += flen

print(f"  input_ids: {input_ids[0].tolist()}")
print(f"  labels:    {labels[0].tolist()}")
print()

# Longer tokens
print("Token-by-token comparison: L=loss, .=ignored")
print(f"  {'Pos':<4s} {'input_id':>8s} {'token':<18s} {'label':>8s} {'Source'}")
print(f"  {'-'*4} {'-'*8} {'-'*18} {'-'*8} {'-'*30}")

pos = 0
for msg in messages:
    role = msg["role"]
    content = msg["content"]

    header_ids = encode(f"{IM_START}{role}\n")
    for hid in header_ids:
        word = id_to_word.get(hid, "???")
        lid = labels[0, pos].item()
        m = "." if lid == IGNORE else "L"
        print(f"  [{pos:<2d}] {hid:>8d} {word:<18s} {lid:>8d} {m} frame({role})")
        pos += 1

    content_ids = encode(content)
    for cid in content_ids:
        word = id_to_word.get(cid, "???")
        lid = labels[0, pos].item()
        m = "L" if lid != IGNORE else "."
        note = f"{m} [{role} content]"
        print(f"  [{pos:<2d}] {cid:>8d} {word:<18s} {lid:>8d} {note}")
        pos += 1

    footer_ids = encode(f"{IM_END}\n")
    for fid in footer_ids:
        word = id_to_word.get(fid, "???")
        lid = labels[0, pos].item()
        m = "." if lid == IGNORE else "L"
        print(f"  [{pos:<2d}] {fid:>8d} {word:<18s} {lid:>8d} {m} frame marker")
        pos += 1

n_compute = (labels != IGNORE).sum().item()
n_ignore = (labels == IGNORE).sum().item()
print(f"\n  Count: {n_compute} tokens contribute loss; {n_ignore} are ignored")
print(f"  Effective training-signal share: {n_compute}/{n_compute+n_ignore} = {n_compute/(n_compute+n_ignore)*100:.1f}%")

# ============================================================
# Step 3: verify that ignore_index really skips positions labeled -100
# ============================================================
print(f"\n{'='*60}")
print("Step 3: verify ignore_index behavior")
print("=" * 60)

# Simulate random model logits
VOCAB_SIZE = max(vocab.values()) + 1
torch.manual_seed(42)
logits = torch.randn(1, input_ids.shape[1], VOCAB_SIZE)

# Method A: use ignore_index, the normal approach
loss_with_ignore = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    labels.reshape(-1),
    ignore_index=IGNORE
)
print(f"\nLoss with ignore_index={IGNORE}: {loss_with_ignore.item():.4f}")

# Method B: verify manually by selecting only labels not equal to IGNORE
losses = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    labels.reshape(-1),
    ignore_index=IGNORE,
    reduction='none'  # Keep per-position losses instead of averaging
)
losses_reshaped = losses.view(input_ids.shape)  # [1, seq_len]

print("\nPer-position loss; ignore_index automatically zeros positions labeled -100:")
for i in range(input_ids.shape[1]):
    tid = input_ids[0, i].item()
    lid = labels[0, i].item()
    pos_loss = losses_reshaped[0, i].item()
    word = id_to_word.get(tid, "???")
    if lid == IGNORE:
        print(f"  [{i:2d}] loss={pos_loss:.4f}, ignored position")
    else:
        print(f"  [{i:2d}] loss={pos_loss:.4f}, valid position predicting {word}")

# Calculate the mean manually
manual_avg = losses_reshaped[labels != IGNORE].mean()
print(f"\nManual mean over valid positions: {manual_avg:.4f}")
print(f"PyTorch cross_entropy result:     {loss_with_ignore.item():.4f}")
print("Match" if abs(manual_avg.item() - loss_with_ignore.item()) < 1e-5 else "Mismatch")

# ============================================================
# Step 4: compare what happens without ignore_index
# ============================================================
print(f"\n{'='*60}")
print("Step 4: compare without ignore_index")
print("=" * 60)

# Replace every -100 label with a real token ID, using 0 as UNK
labels_bad = labels.clone()
labels_bad[labels_bad == IGNORE] = 0

loss_bad = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    labels_bad.reshape(-1)
)
print(f"Loss after replacing every -100 with 0: {loss_bad.item():.4f}")
print(f"Correct loss, ignoring -100:             {loss_with_ignore.item():.4f}")
print(f"Difference: {abs(loss_bad.item() - loss_with_ignore.item()):.4f}")
print()
print("Without ignore_index, the model is forced to learn meaningless targets:")
print("  predict <|im_end|> after the system prompt")
print("  predict <|im_end|> after the user question")
print("  Those targets add noise rather than teaching the answer.")
print()
print('"Summary:"')
print("  1. Assistant-content labels use real token IDs, so they contribute loss and teach answer generation")
print("  2. Every other label is -100, so the model can read those tokens without learning to reproduce them")
print("  3. CrossEntropyLoss(ignore_index=-100) skips them automatically, producing zero loss and gradient there")


### 10.4 Multi-Turn Conversations

Real training data often contains multi-turn conversations. For example:
```json
{"messages": [
    {"role": "system", "content": "You are a math teacher"},
    {"role": "user", "content": "1+1=?"},
    {"role": "assistant", "content": "1+1=2"},
    {"role": "user", "content": "What about 2+2?"},
    {"role": "assistant", "content": "2+2=4"}
]}
```

**The concatenation rule is exactly the same**: all messages are concatenated in order into a single continuous token sequence.

```
<|im_start|>system\nYou are a math teacher<|im_end|>\n
<|im_start|>user\n1+1=?<|im_end|>\n
<|im_start|>assistant\n1+1=2<|im_end|>\n
<|im_start|>user\nWhat about 2+2?<|im_end|>\n
<|im_start|>assistant\n2+2=4<|im_end|>\n
```

**Labels are also the same**: each turn's assistant content participates in loss computation; system/user/special markers are all ignored.

```
+------------------------------------------------------------------+
|              Multi-turn conversation labels rule                    |
+------------------------------------------------------------------+
|                                                                    |
|  Turn 1: user->"1+1=?"        <- model reads, doesn't learn       |
|           assistant->"1+1=2"   <- model reads, must learn!        |
|                                                                    |
|  Turn 2: user->"What about 2+2?" <- model reads, doesn't learn    |
|           assistant->"2+2=4"   <- model reads, must learn!        |
|                                                                    |
|  Through attention, the model can see the full history from Turn 1 |
|  -> the model learns to "answer based on conversation history"     |
|                                                                    |
+------------------------------------------------------------------+
```

**Why is multi-turn data important?**
- Single-turn "one question, one answer" -> model can only give one response
- Multi-turn "continuous conversation" -> model learns: follow-up questions, clarification, remembering context
- Real training typically mixes: ~60% multi-turn + ~40% single-turn

Below is a demo of multi-turn conversation concatenation and label construction:

In [ ]:
# ============================================================
# Prove that five messages in a multi-turn dialogue still concatenate into one sequence
# ============================================================

print("=" * 70)
print("Proof: a multi-turn dialogue is one continuous token sequence")
print("=" * 70)
print()

# Simulated vocabulary
vocab = {
    "<|im_start|>": 151644, "<|im_end|>": 151645,
    "system": 8948, "user": 872, "assistant": 78191,
    "you": 9942, "are": 10603, "math": 107659, "teacher": 113740,
    "1": 16, "+": 17, "2": 18, "=": 19, "?": 20, "。": 21,
    "3": 22, "4": 23, "then": 104322, "what": 104535, "\n": 198,
}
id_to_word = {v: k for k, v in vocab.items()}

def encode(text):
    tokens = []
    i = 0
    while i < len(text):
        matched = None
        for word in sorted(vocab.keys(), key=lambda x: -len(x)):
            if text[i:].startswith(word):
                matched = word
                break
        if matched:
            tokens.append(vocab[matched])
            i += len(matched)
        else:
            tokens.append(0)
            i += 1
    return tokens

IM_START = "<|im_start|>"
IM_END = "<|im_end|>"
IGNORE = -100

# Two-turn dialogue data
multi_turn = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant that extracts information. Answer briefly based on the text above."},
        {"role": "user", "content": "1+1=?"},
        {"role": "assistant", "content": "1+1=2。"},
        {"role": "user", "content": "Then what is 2+2?"},
        {"role": "assistant", "content": "2+2=4。"},
    ]
}

print("Multi-turn data: five messages = system plus two user-assistant turns")
for i, msg in enumerate(multi_turn["messages"]):
    print(f"  [{i}] {msg['role']:>10s}: {msg['content']}")

# ============================================================
# Concatenate all five messages step by step
# ============================================================
print()
print("=" * 70)
print("Step-by-step concatenation of all five messages")
print("=" * 70)

all_text = ""
all_ids = []

for step, msg in enumerate(multi_turn["messages"]):
    role = msg["role"]
    content = msg["content"]

    segment = f"{IM_START}{role}\n{content}{IM_END}\n"
    segment_ids = encode(segment)

    before_len = len(all_ids)
    all_text += segment
    all_ids.extend(segment_ids)

    print(f"\nStep {step+1}/5: append {role} -> {content}")
    print(f"  Segment: {repr(segment)}")
    print(f"Segment token count: {len(segment_ids)}")
    print(f"Cumulative token count: {before_len} -> {len(all_ids)}")
    print(f"Cumulative text: {repr(all_text)}")

# ============================================================
# Final proof
# ============================================================
print(f"\n{'='*70}")
print("Final result: five messages form one continuous token sequence")
print(f"{'='*70}")
print(f"  Total tokens: {len(all_ids)}")
print(f"  Complete IDs: {all_ids}")
print()

# Annotate each token with its turn
print("Token-by-token annotation proving that the sequence is continuous:")
print(f"{'Pos':<4s} {'ID':>7s} {'token':<16s} {'Turn/role':<25s} {'Loss?'}")
print(f"{'-'*4} {'-'*7} {'-'*16} {'-'*25} {'-'*8}")

pos = 0
for turn_idx, msg in enumerate(multi_turn["messages"]):
    role = msg["role"]
    content = msg["content"]

    header_ids = encode(f"{IM_START}{role}\n")
    for hid in header_ids:
        word = id_to_word.get(hid, "???")
        print(f"{pos:<4d} {hid:>7d} {word:<16s} {'turn '+str(turn_idx+1)+' frame('+role+')':<25s} ignored")
        pos += 1

    content_ids = encode(content)
    for cid in content_ids:
        word = id_to_word.get(cid, "???")
        if role == "assistant":
            loss_note = "compute"
        else:
            loss_note = "ignore"
        print(f"{pos:<4d} {cid:>7d} {word:<16s} {'turn '+str(turn_idx+1)+' '+role+' content':<25s} {loss_note}")
        pos += 1

    footer_ids = encode(f"{IM_END}\n")
    for fid in footer_ids:
        word = id_to_word.get(fid, "???")
        print(f"{pos:<4d} {fid:>7d} {word:<16s} {'turn '+str(turn_idx+1)+' frame end':<25s} ignored")
        pos += 1

    if turn_idx < len(multi_turn["messages"]) - 1:
        print(f"{'':>4s} {'continue concatenating without a break':>50s}")

print()
print(f"Current config:")
print(f"  Five independent messages become one continuous sequence of {pos} tokens")
print("  The model reads the entire sequence at once with causal attention")
print("  Only assistant-content tokens contribute to loss")
print("  Multi-turn data simply repeats the same concatenation rule used for one turn")


### 10.5 A Complete Training Loop for Conversation Data

Recall the loop from Section 6: `input_ids = batch[:, :-1]` and `labels = batch[:, 1:]`. Conversation training uses the same sequence shift, but applies a label mask so only selected assistant tokens contribute to loss.

```text
1. Format messages with a Chat Template
2. Tokenize the formatted conversation
3. Prepare input and shifted labels
4. Replace masked label positions with -100
5. Forward -> loss -> backward -> optimizer step
```

### 10.6 Connecting Training and Generation

During training, teacher forcing supplies the full shifted sequence so every supervised position is predicted in parallel. During inference, the model begins with the formatted prompt and enters autoregressive generation, appending one sampled token at a time.


In [ ]:
# ============================================================
# Complete training loop: Chat Template plus MiniGPT
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=== Complete training loop: dialogue data to tokens to training ===\n")

# Reuse the vocabulary and helper functions defined above
vocab = {
    "<|im_start|>": 151644, "<|im_end|>": 151645,
    "system": 8948, "user": 872, "assistant": 78191,
    "you": 9942, "are": 10603, "math": 107659, "teacher": 113740,
    "1": 16, "+": 17, "2": 18, "=": 19, "?": 20, "。": 21,
    "3": 22, "4": 23, "then": 104322, "what": 104535, "\n": 198,
    "translate": 112345, "officer": 105678,
    "Hello": 201, '"hello"': 202, "!": 203,
    "weather": 301, "how": 302, "is": 303,
    "today": 304, "not": 305, "bad": 306, "sunny": 307,
}
id_to_word = {v: k for k, v in vocab.items()}
VOCAB_SIZE = max(vocab.values()) + 10
IM_START = "<|im_start|>"
IM_END = "<|im_end|>"
IGNORE = -100
PAD_ID = 0

def encode(text):
    tokens = []
    i = 0
    while i < len(text):
        matched = None
        for word in sorted(vocab.keys(), key=lambda x: -len(x)):
            if text[i:].startswith(word):
                matched = word
                break
        if matched:
            tokens.append(vocab[matched])
            i += len(matched)
        else:
            tokens.append(0)
            i += 1
    return tokens

# Prepare three training conversations
train_conversations = [
    {
        "messages": [
            {"role": "system", "content": "you are math teacher"},
            {"role": "user", "content": "1+1=?"},
            {"role": "assistant", "content": "1+1=2。"},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "you are translate officer"},
            {"role": "user", "content": "Hello!"},
            {"role": "assistant", "content": '"hello!"'},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "weather how is?"},
            {"role": "assistant", "content": "today weather not bad sunny."},
        ]
    },
]

# Construct input_ids and labels
print("=== Construct training data ===")
all_inputs = []
all_labels = []

for idx, conv in enumerate(train_conversations):
    messages = conv["messages"]

    # Concatenate message text
    text = ""
    for msg in messages:
        text += f"{IM_START}{msg['role']}\n{msg['content']}{IM_END}\n"

    input_ids = encode(text)
    labels = [IGNORE] * len(input_ids)

    # Label assistant-content positions
    pos = 0
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        pos += len(encode(f"{IM_START}{role}\n"))
        cids = encode(content)
        if role == "assistant":
            for j, cid in enumerate(cids):
                labels[pos + j] = cid
        pos += len(cids)
        pos += len(encode(f"{IM_END}\n"))

    all_inputs.append(input_ids)
    all_labels.append(labels)

    n_assistant = sum(1 for l in labels if l != IGNORE)
    n_total = len(labels)
    print(f"Dialogue {idx+1}: {n_total} tokens, {n_assistant} contribute loss ({n_assistant/n_total*100:.0f}%)")
    # Annotate conversations without a system prompt
    has_system = any(m["role"] == "system" for m in messages)
    if not has_system:
        print("  no system prompt")

print()

# Pad every sequence to the same length
max_len = max(len(ids) for ids in all_inputs)
print(f"Maximum sequence length: {max_len}")

padded_inputs = []
padded_labels = []

for input_ids, labels in zip(all_inputs, all_labels):
    pad_len = max_len - len(input_ids)
    padded_inputs.append(input_ids + [PAD_ID] * pad_len)
    padded_labels.append(labels + [IGNORE] * pad_len)

input_ids_batch = torch.tensor(padded_inputs)
labels_batch = torch.tensor(padded_labels)

print(f"input_ids_batch shape: {input_ids_batch.shape}")
print(f"labels_batch shape: {labels_batch.shape}")
print()
print("input_ids_batch:")
print(input_ids_batch)
print()
print("labels_batch, where -100 means ignored:")
print(labels_batch)
print()

# Split into model inputs and targets
# As earlier in the chapter, remove the final input token and the first target token
model_inputs = input_ids_batch[:, :-1]   # [batch, seq-1]
model_targets = labels_batch[:, 1:]       # [batch, seq-1]

print(f"Model-input shape: {model_inputs.shape}")
print(f"Model-target shape: {model_targets.shape}")
print()

# Create a small model
class TinyLLM(nn.Module):
    def __init__(self, vocab_size, d_model=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead=4, dim_feedforward=64, batch_first=True),
            num_layers=2
        )
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        mask = nn.Transformer.generate_square_subsequent_mask(x.shape[1], device=x.device)
        x = self.embed(x)
        x = self.transformer(x, mask=mask, is_causal=True)
        return self.lm_head(x)

model = TinyLLM(VOCAB_SIZE, d_model=32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print()

# Training loop
NUM_EPOCHS = 10
losses = []

print(f"=== Train for {NUM_EPOCHS} epochs ===")
model.train()
for epoch in range(NUM_EPOCHS):
    logits = model(model_inputs)  # [batch=3, seq-1, vocab_size]

    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        model_targets.reshape(-1),
        ignore_index=IGNORE  # Crucial: skip positions labeled -100
    )

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    losses.append(loss.item())
    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1:2d}/{NUM_EPOCHS} | Loss: {loss.item():.4f}")

print(f"\nLoss: {losses[0]:.4f} -> {losses[-1]:.4f}; falling loss means learning assistant-response patterns")
print()

# Inference demonstration with the trained model
print("=== Inference demonstration ===")
# Construct a prompt containing system and user messages
prompt_messages = [
    {"role": "system", "content": "you are math teacher"},
    {"role": "user", "content": "2+2=?"}
]

# Concatenate the prompt with the same chat template but a different ending
prompt_text = ""
for msg in prompt_messages:
    prompt_text += f"{IM_START}{msg['role']}\n{msg['content']}{IM_END}\n"
prompt_text += f"{IM_START}assistant\n"  # ← add_generation_prompt

prompt_ids = torch.tensor([encode(prompt_text)])
print(f"Prompt: {prompt_text.strip()}")
print(f"Prompt IDs: {prompt_ids[0].tolist()}")

# Autoregressive generation
model.eval()
generated = prompt_ids.clone()
with torch.no_grad():
    for _ in range(10):  # Generate at most ten tokens
        logits = model(generated)
        next_logits = logits[0, -1, :]  # Prediction from the final position
        probs = F.softmax(next_logits / 0.7, dim=-1)

        # Forbid PAD and special tokens during this teaching example
        probs[0] = 0
        probs[151644] = 0
        probs[151645] = 0

        next_token = torch.multinomial(probs, 1)
        generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)

        if next_token.item() == 151645:  # <|im_end|>
            break

# Decode generated IDs
output_ids = generated[0].tolist()
output_text = ""
for tid in output_ids[len(prompt_ids[0]):]:  # Keep only newly generated tokens
    word = id_to_word.get(tid, f"[{tid}]")
    output_text += word
    if tid == 151645:
        break

print(f"Generated text: {output_text}")
print()

# Summary
print("=" * 60)
print("Complete pipeline recap:")
print("=" * 60)
print("JSONL -> messages -> Chat Template -> input_ids -> assistant-only labels -> cross-entropy training -> autoregressive generation")


## 11. Warmup

When the model is first initialized, parameters are random. If we use a large learning rate (e.g., 0.01) from the start, gradient directions are chaotic, parameters get "pulled all over the place," and loss may explode.

**Warmup approach**: for the first N steps (usually 5% of total steps), linearly increase the learning rate from 0 to the target value. Let the model "probe" with small steps first, find roughly the right direction, then accelerate.

```
  LR
  |        /----------------------  <- normal training
  |      /
  |    /
  |  /
  |/  <- Warmup phase (first 5% of steps)
  +-------------------------------> Step
```

Real-world intuition for warmup: starting a car in winter, idle for 30 seconds before driving -- flooring the gas immediately would damage the engine.

In [ ]:
# === Hand-calculate and visualize warmup ===
import matplotlib.pyplot as plt

import math

print("=== SwiGLU hand calculation ===")
print()

total_steps = 1000
warmup_steps = 50   # Warm up over the first fifty steps
max_lr = 0.01

def warmup_lr(step, warmup_steps, max_lr, total_steps):
    """Linear warmup followed by cosine decay."""
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    else:
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return max_lr * 0.5 * (1 + math.cos(math.pi * progress))

# Learning rates at representative steps
check_points = [0, 10, 25, 50, 100, 500, 900, 999]
print(f"Total steps: {total_steps}, warmup steps: {warmup_steps}, maximum LR: {max_lr}")
print()
print(f"{'Step':>6s}  {'LR':>12s}  {'Phase'}")
print('-' * 40)
for step in check_points:
    lr = warmup_lr(step, warmup_steps, max_lr, total_steps)
    phase = "Warmup" if step < warmup_steps else "Training"
    print(f"{step:>6d}  {lr:>12.6f}  {phase}")

print()
print('"Key observation:"')
print("  Step 0: LR = 0, so parameters do not move before warming begins")
print(f"  Step 10: LR = {warmup_lr(10, warmup_steps, max_lr, total_steps):.6f}, rising gradually")
print(f"  Step 50: LR = {warmup_lr(50, warmup_steps, max_lr, total_steps):.6f}, maximum reached")
print(f"  Step 500: LR = {warmup_lr(500, warmup_steps, max_lr, total_steps):.6f}, during cosine decay")

# Visualization
steps = list(range(total_steps))
lrs = [warmup_lr(s, warmup_steps, max_lr, total_steps) for s in steps]

plt.figure(figsize=(10, 3))
plt.plot(steps, lrs, linewidth=1.5)
plt.axvspan(0, warmup_steps, alpha=0.2, color='orange', label='Warmup')
plt.axvspan(warmup_steps, total_steps, alpha=0.05, color='blue', label='Cosine Decay')
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.title('Warmup + Cosine Decay for LLM Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Orange is linear warmup; blue is ordinary training under cosine decay")
print("The scaling-laws chapter also introduces WSD, which replaces cosine decay with a stable phase before decay")


## 12. Multi-Token Prediction

Standard autoregressive training predicts one next token at each position. Multi-Token Prediction (MTP) adds auxiliary heads that predict several future tokens, increasing the density of the training signal.

For ordinary autoregressive generation, only `Head_1` may be retained and the auxiliary heads discarded; in that case MTP's benefit comes mainly from training. DeepSeek-V3 also shows that the MTP module can serve as a draft module for speculative decoding: auxiliary heads propose future tokens and the main model verifies them. The precise statement is therefore: **MTP may be removed for ordinary decoding or reused by a compatible speculative-decoding engine.** See [DeepSeek-V3 GitHub](https://github.com/deepseek-ai/DeepSeek-V3) and the [technical report](https://arxiv.org/abs/2412.19437).


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadLM(nn.Module):
    """Several output heads that predict tokens one through N steps into the future."""
    def __init__(self, d_model, vocab_size, num_heads=4):
        super().__init__()
        self.num_heads = num_heads

        self.heads = nn.ModuleList([
            nn.Linear(d_model, vocab_size, bias=False)
            for _ in range(num_heads)
        ])

    def forward(self, hidden_states):
        """Return one logits tensor per prediction head for hidden states [batch, seq_len, d_model]."""
        return [head(hidden_states) for head in self.heads]

def compute_mtp_loss(logits_list, target_ids, ignore_index=-100):
    """Average MTP loss after aligning head i with the token i+1 steps ahead."""
    total_loss = 0.0
    for i, logits in enumerate(logits_list):
        shift = i + 1
        # Remove the final shift positions because they have no corresponding target
        logits_trimmed = logits[:, :-shift, :]
        targets = target_ids[:, shift:]

        logits_flat = logits_trimmed.reshape(-1, logits.shape[-1])
        targets_flat = targets.reshape(-1)

        total_loss += F.cross_entropy(logits_flat, targets_flat,
                                      ignore_index=ignore_index)
    return total_loss / len(logits_list)

print("Multi-Token Prediction components are defined.")
print("Each head predicts a different future distance; inference retains only the main head.")


In [ ]:
# Demonstrate MTP loss calculation
import torch
import torch.nn.functional as F

torch.manual_seed(42)

V = 20
B, S, D = 2, 8, 32

# Simulate main-model hidden states and correct target IDs
hidden = torch.randn(B, S, D)
targets = torch.randint(0, V, (B, S))

mtp = MultiHeadLM(D, V, num_heads=4)
logits_list = mtp(hidden)

print("=== Multi-Token Prediction loss demonstration ===")
print(f"Input shape: batch={B}, seq_len={S}, d_model={D}")
print(f"Prediction heads: {len(logits_list)}")
print()

# Compare standard single-head training with MTP
print("Standard single-head training:")
single_loss = F.cross_entropy(
    logits_list[0][:, :-1, :].reshape(-1, V),
    targets[:, 1:].reshape(-1)
)
print(f"  Head 0 alone predicts t+1, loss = {single_loss.item():.4f}")
print("  Each token receives one supervision signal")

print()
print("MTP training with four heads:")
for i, logits in enumerate(logits_list):
    shift = i + 1
    effective = S - shift
    head_loss = F.cross_entropy(
        logits[:, :-shift, :].reshape(-1, V),
        targets[:, shift:].reshape(-1)
    )
    print(f"  Head {i}, predicting t+{shift}: valid positions={effective}, loss={head_loss.item():.4f}")

total = compute_mtp_loss(logits_list, targets)
print(f"  Mean total MTP loss: {total.item():.4f}")

print()
print("Key observation:")
print("1. Head 0 is ordinary next-token prediction, identical to single-head training")
print("2. Heads 1-3 predict farther tokens and have fewer valid tail positions")
print("3. One hidden-state sequence receives four supervision signals, increasing training information density")
print("4. Inference keeps Head 0 and discards Heads 1-3, leaving inference speed unchanged")


## 13. The Hugging Face Transformers Training Interface

We have already expanded the loop down to each parameter update. Transformers packages the same components behind `Trainer` and `TrainingArguments`.

| Training component | Transformers component |
|:---|:---|
| Token sequences | `Dataset` or `datasets.Dataset` |
| Assemble samples into batches | `DataCollatorForLanguageModeling` or a custom collator |
| Parameter updates and learning rate | Managed inside Trainer |
| Epoch and batch loops | `trainer.train()` |
| Gradient accumulation | Configured by TrainingArguments |

`trainer.train()` organizes operations we already implemented; it does not replace their mathematics.


## 14. The ModelScope SWIFT Training Interface

The same SFT task can be configured with **ModelScope SWIFT** (`ms-swift`). It packages model loading, Chat Templates, LoRA, distributed settings, and evaluation as command-line arguments.

| Option | Meaning in the training loop |
|:---|:---|
| `--per_device_train_batch_size` | Samples processed by each device per small batch |
| `--learning_rate` | Step size for each parameter update |
| `--deepspeed` and distributed options | Shard model, gradients, and training state across devices |
| `--output_dir` | Store checkpoints and logs |

SWIFT changes how training is organized, not the mathematics of loss, backward, and parameter updates.


## 15. Label Shifting

| External data format | Where shifting happens | Typical use |
|:---|:---|:---|
| `input=tokens[:-1]`, `labels=tokens[1:]` | In the data pipeline | Explicit teaching implementations |
| `input=tokens`, `labels=tokens` | Inside the model's loss calculation | Common training interfaces |

Even when code passes `labels=input_ids`, the model internally compares logits at the current position with the label of the next token.


In [ ]:
# Compare: external shift in the data vs internal shift in the model — same supervision
full_tokens = torch.tensor([[1, 3, 4, 5, 6, 2]])

external_input = full_tokens[:, :-1]
external_labels = full_tokens[:, 1:]

internal_input = full_tokens
internal_labels = full_tokens
shifted_labels_inside_model = internal_labels[:, 1:]

print("External shift:")
print("input: ", external_input.tolist())
print("labels:", external_labels.tolist())
print()

print("Equivalent result of internal shift:")
print("input passes full tokens:    ", internal_input.tolist())
print("model takes labels[:, 1:]:", shifted_labels_inside_model.tolist())
print()

same = torch.equal(external_labels, shifted_labels_inside_model)
print("Do both forms give the same prediction target?", same)
print("Key observation: industrial libraries often hide the shift inside the model's loss computation.")


## Summary

- [ ] I can construct shifted input and labels for next-token prediction.
- [ ] I can explain token-level Cross-Entropy and masking.
- [ ] I understand gradients, clipping, accumulation, and warmup.
- [ ] I can distinguish training-time teacher forcing from autoregressive inference.
- [ ] I know how a Chat Template and label mask supervise assistant responses.
- [ ] I understand that `trainer.train()` organizes data, forward, backward, and updates.
- [ ] I know that a Causal LM may shift logits and labels internally.
- [ ] I can explain how MTP adds future-token supervision and may support speculative decoding.


## Exercises

> You can ask an AI to explain the ideas or check your direction, but don't have it "solve the exercise" for you.

**Exercise 1: Construct Training Input and Target**

For the token sequence `[0, 5, 3, 8, 2]`, where 0 is BOS and 2 is EOS, write the full input and target sequences used by an autoregressive language model.

Hint: `input = sentence[:-1]` and `target = sentence[1:]`.


In [ ]:
# Exercise 1: construct training inputs and targets
sentence = [0, 5, 3, 8, 2]  # [BOS, I, love, NLP, EOS]
# TODO: construct the input by removing the final token
input_ids = None  # Construct it here
# TODO: construct the target by removing the first token
target_ids = None  # Construct it here
assert input_ids is not None, "Construct input_ids first"
assert target_ids is not None, "Construct target_ids first"
assert input_ids == [0, 5, 3, 8], f"Expected input [0, 5, 3, 8], got {input_ids}"
assert target_ids == [5, 3, 8, 2], f"Expected target [5, 3, 8, 2], got {target_ids}"
print("Exercise 1 passed:")
print(f"   Original: {sentence}")
print(f"   Input:    {input_ids}, what the model sees")
print(f"   Target:   {target_ids}, what the model predicts")
print("   Every position predicts the next token.")


**Exercise 2: Calculate Cross-Entropy by Hand**

For logits `[2.0, 1.0, 0.1]` and correct class 0, calculate the softmax probabilities and then $-\log(p_{correct})$.

Hint: $p_i=e^{z_i}/\sum_j e^{z_j}$ and loss $=-\log(p_0)$.


In [ ]:
# Exercise 2: hand-calculate Cross-Entropy Loss
import math
logits = [2.0, 1.0, 0.1]
correct_class = 0
# TODO: calculate softmax
# p_i = exp(logits[i]) / sum(exp(logits))
exp_vals = None  # First calculate every exponential
probs = None     # Then calculate probabilities
# TODO: calculate loss = -log(probs[correct_class])
loss = None  # Calculate it here
assert exp_vals is not None, "Calculate the exponential values first"
assert probs is not None, "Calculate probabilities first"
assert loss is not None, "Calculate loss first"
assert abs(sum(probs) - 1.0) < 0.001, f"Probabilities should sum to 1.0, got {sum(probs):.4f}"
expected_loss = -math.log(probs[0])
assert abs(loss - expected_loss) < 0.001, f"Expected loss {expected_loss:.4f}"
import torch
import torch.nn.functional as F
torch_loss = F.cross_entropy(torch.tensor([logits]), torch.tensor([correct_class]))
assert abs(loss - torch_loss.item()) < 0.01, f"Does not match PyTorch: {torch_loss.item():.4f}"
print("Exercise 2 passed:")
print(f"   Softmax probabilities: [{probs[0]:.4f}, {probs[1]:.4f}, {probs[2]:.4f}]")
print(f"   Loss = -log({probs[0]:.4f}) = {loss:.4f}")
print(f"   PyTorch check: {torch_loss.item():.4f}")
print("   Higher probability on the correct class produces lower loss.")


**Exercise 3: Padding Loss Mask**

Targets are `[[5, 3, 8, 2], [7, 4, -100, -100]]`, and losses at the eight positions are `[0.5, 0.3, 0.8, 0.2, 0.6, 0.4, 1.0, 0.9]`. Calculate mean loss over valid positions only.

Hint: six positions are valid; CrossEntropyLoss ignores labels equal to `-100`.


In [ ]:
# Exercise 3: mask PAD positions out of loss
# Losses at eight positions: two samples times four positions, flattened by row
all_losses = [0.5, 0.3, 0.8, 0.2, 0.6, 0.4, 1.0, 0.9]
# PAD flags for the final two positions of the second sample
is_pad = [False, False, False, False, False, False, True, True]
# TODO: retain only non-PAD losses and calculate their mean
valid_loss = None  # Calculate it here
assert valid_loss is not None, "Calculate valid_loss first"
valid_losses = [l for l, pad in zip(all_losses, is_pad) if not pad]
expected = sum(valid_losses) / len(valid_losses)
assert abs(valid_loss - expected) < 0.001, f"Expected valid loss {expected:.4f}, got {valid_loss:.4f}"
print("Exercise 3 passed:")
print(f"   All position losses: {all_losses}")
print(f"   Valid position losses: {valid_losses}")
print(f"   Mean valid loss: {valid_loss:.4f}")
print("   PAD positions do not contribute loss, so meaningless fill tokens do not disturb training.")


**Exercise 4: Complete a TrainingArguments Configuration**

Configure two samples per device, accumulation over eight small batches, logging every 10 steps, and checkpointing every 500 steps.

Hint: use `per_device_train_batch_size`, `gradient_accumulation_steps`, `logging_steps`, and `save_steps`.


In [ ]:
# Exercise 4: complete a TrainingArguments configuration

# TODO: map the four training requirements to fields with the same names as TrainingArguments
training_args = {
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "logging_steps": 10,
    "save_steps": 500,
}

assert training_args["per_device_train_batch_size"] == 2
assert training_args["gradient_accumulation_steps"] == 8
assert training_args["logging_steps"] == 10
assert training_args["save_steps"] == 500

print("Exercise 4 passed: the requirements are mapped to TrainingArguments fields.")


**Exercise 5: Effective Batch Size**

Training parameters:

```text
per_device_train_batch_size = 2
num_gpus = 4
gradient_accumulation_steps = 8
```


Compute how many samples one optimizer step effectively sees.

Hint: effective batch size = per-GPU batch × number of GPUs × gradient accumulation steps.


In [ ]:
# Exercise 5: calculate effective batch size
per_device_train_batch_size = 2
num_gpus = 4
gradient_accumulation_steps = 8

# TODO: calculate how many samples accumulate before one optimizer.step()
effective_batch_size = per_device_train_batch_size * num_gpus * gradient_accumulation_steps

assert effective_batch_size is not None, 'Please replace the placeholder before running the assertion.'
assert effective_batch_size == 64, f"Expected 64, got {effective_batch_size}"

print("Exercise 5 passed:")
print(f"   Effective batch size = {effective_batch_size}")
print("   This is why gradient accumulation is commonly used when memory is limited.")
